# SafeRL-Drive Phase-2 Colab driver

This notebook closes Phase 1 and runs the approved Phase-2 direct-versus-curriculum SAC study. GitHub and `/content/safedrive` hold live code; `/content/drive/MyDrive/SafeDrive` holds persistent artifacts. Each long curriculum stage is copied to Drive before the next begins. Google Drive mounting is sufficient, so PyDrive is not required.

Run sections in order. Confirmation seeds are deliberately blocked unless the seed-0 pilots meet the preregistered promising-result gate.

## 1. Initialize paths and experiment constants

In [1]:
from pathlib import Path
from datetime import datetime, timezone
import json
import os
import shutil
import subprocess
import sys

REPO_URL = "https://github.com/djdhillxn/safedrive.git"
REPO_DIR = Path("/content/safedrive")
DRIVE_MOUNT = Path("/content/drive")
DRIVE_PROJECT = DRIVE_MOUNT / "MyDrive" / "SafeDrive"
PHASE1_TEST_EPISODES = 20
PHASE2_TIMESTEPS = 500_000
PHASE2_TEST_EPISODES = 100
VIDEO_STEPS = 1_000
CONFIRMATION_SEEDS = [1, 2]

print(f"Live checkout: {REPO_DIR}")
print(f"Persistent artifacts: {DRIVE_PROJECT}")
print(f"Phase-2 maximum per condition: {PHASE2_TIMESTEPS:,} steps")

Live checkout: /content/safedrive
Persistent artifacts: /content/drive/MyDrive/SafeDrive
Phase-2 maximum per condition: 500,000 steps


## 2. Runtime and GPU check

In [2]:
import torch

print(f"Python: {sys.version.split()[0]}")
if shutil.which("nvidia-smi"):
    subprocess.run(["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"], check=False)
else:
    print("nvidia-smi: unavailable")
print(f"PyTorch CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

Python: 3.12.13
PyTorch CUDA: True
NVIDIA L4


## 3. Mount Google Drive

In [4]:
from google.colab import drive

drive.mount(str(DRIVE_MOUNT))
DRIVE_PROJECT.mkdir(parents=True, exist_ok=True)
connection_file = DRIVE_PROJECT / "colab_connection_test.txt"
connection_file.write_text(f"Connected at {datetime.now(timezone.utc).isoformat()}\n", encoding="utf-8")
print(f"Drive read/write ready: {connection_file}")

Mounted at /content/drive
Drive read/write ready: /content/drive/MyDrive/SafeDrive/colab_connection_test.txt


## 4. Clone or fast-forward pull the public repository

In [5]:
if (REPO_DIR / ".git").exists():
    git_command = ["git", "-C", str(REPO_DIR), "pull", "--ff-only"]
elif REPO_DIR.exists() and any(REPO_DIR.iterdir()):
    raise RuntimeError(f"{REPO_DIR} exists but is not a Git repository.")
else:
    if REPO_DIR.exists():
        REPO_DIR.rmdir()
    git_command = ["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)]
git_result = subprocess.run(git_command, capture_output=True, text=True)
if git_result.returncode:
    print(git_result.stdout[-4000:])
    print(git_result.stderr[-4000:])
    raise RuntimeError("Git clone/pull failed.")
os.chdir(REPO_DIR)
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))
git_commit = subprocess.check_output(["git", "-C", str(REPO_DIR), "rev-parse", "HEAD"], text=True).strip()
print(f"Repository ready at commit {git_commit[:12]}")

Repository ready at commit b2d588885073


## 4.1 Restore persistent artifacts from Drive

Full restore is intentional in Colab because evaluation, video, and curriculum resume need model files. The default Mac sync remains analysis-only.

In [6]:
subprocess.run([sys.executable, "-m", "scripts.sync_drive_runs", "--drive-project", str(DRIVE_PROJECT), "--local-runs", str(REPO_DIR / "runs"), "--include-training-artifacts"], cwd=REPO_DIR, check=True)

CompletedProcess(args=['/usr/bin/python3', '-m', 'scripts.sync_drive_runs', '--drive-project', '/content/drive/MyDrive/SafeDrive', '--local-runs', '/content/safedrive/runs', '--include-training-artifacts'], returncode=0)

## 5. Install SafeRL-Drive

In [7]:
import importlib.metadata

subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "gym"], check=False, capture_output=True, text=True)
install_result = subprocess.run([sys.executable, "-m", "pip", "install", "--disable-pip-version-check", "--progress-bar", "off", "-e", "."], cwd=REPO_DIR, capture_output=True, text=True)
if install_result.returncode:
    print(install_result.stdout[-8000:])
    print(install_result.stderr[-8000:])
    raise RuntimeError("SafeRL-Drive installation failed.")
packages = ["metadrive-simulator", "stable-baselines3", "gymnasium", "torch"]
print(json.dumps({name: importlib.metadata.version(name) for name in packages}, indent=2))

{
  "metadrive-simulator": "0.4.3",
  "stable-baselines3": "2.9.0",
  "gymnasium": "1.3.0",
  "torch": "2.11.0+cu128"
}


### Shared artifact, gate, and resume helpers

In [8]:
def read_latest_run(name):
    pointer = REPO_DIR / "runs" / f"latest_{name}.txt"
    if not pointer.exists():
        raise FileNotFoundError(f"Latest-run pointer not found: {pointer}")
    run_dir = Path(pointer.read_text(encoding="utf-8").strip())
    if not run_dir.is_absolute():
        run_dir = REPO_DIR / run_dir
    return run_dir.resolve()

def newest_run_containing(text):
    candidates = [path for path in (REPO_DIR / "runs").iterdir() if path.is_dir() and text in path.name]
    if not candidates:
        raise FileNotFoundError(f"No run directory contains {text!r}.")
    return max(candidates, key=lambda path: path.name)

def require_shell_success(label):
    exit_code = get_ipython().user_ns.get("_exit_code", 0)
    if exit_code:
        raise RuntimeError(f"{label} failed with exit code {exit_code}.")

def show_json(path):
    path = Path(path)
    data = json.loads(path.read_text(encoding="utf-8"))
    print(json.dumps(data, indent=2))
    return data

def copy_to_drive(path, category="runs"):
    path = Path(path)
    target = DRIVE_PROJECT / category / path.name
    target.parent.mkdir(parents=True, exist_ok=True)
    if path.is_dir():
        target.mkdir(parents=True, exist_ok=True)
        if shutil.which("rsync"):
            subprocess.run(["rsync", "-a", f"{path}/", f"{target}/"], check=True)
        else:
            shutil.copytree(path, target, dirs_exist_ok=True)
    else:
        shutil.copy2(path, target)
    print(f"Copied to Drive: {target}")
    return target

def backup_pointer_run(name):
    run_dir = read_latest_run(name)
    copy_to_drive(run_dir)
    copy_to_drive(REPO_DIR / "runs" / f"latest_{name}.txt")
    return run_dir

def require_metrics(summary, label, success=0.80, route=0.90, failures=0.10):
    values = {
        "success": float(summary.get("success_rate", 0.0)),
        "route": float(summary.get("mean_route_completion", 0.0)),
        "collision": float(summary.get("collision_rate", 1.0)),
        "off_road": float(summary.get("out_of_road_rate", 1.0)),
        "timeout": float(summary.get("timeout_or_max_step_rate", 1.0)),
    }
    passed = values["success"] >= success and values["route"] >= route and max(values["collision"], values["off_road"], values["timeout"]) <= failures
    print(label, {key: f"{value:.1%}" for key, value in values.items()}, "PASS" if passed else "FAIL")
    return passed

def require_curriculum_state(run_dir, expected):
    state = show_json(Path(run_dir) / "curriculum_state.json")
    if state.get("status") not in expected:
        raise RuntimeError(f"Curriculum status {state.get('status')!r}; expected {expected}.")
    return state

def representative_seed(run_dir):
    import pandas as pd
    frame = pd.read_csv(Path(run_dir) / "eval" / "best_test_episodes.csv")
    success_mask = frame["success"].astype(str).str.lower().eq("true")
    successful = frame[success_mask]
    if not successful.empty:
        return int(successful.iloc[0]["env_seed"])
    best = frame.loc[frame["route_completion"].astype(float).idxmax()]
    print(f"No successful rollout in {run_dir}; recording the highest-completion failure.")
    return int(best["env_seed"])

## 6. Lightweight wiring test

In [9]:
!python -m scripts.train --config configs/smoke_test.yaml
require_shell_success("Smoke test")
SMOKE_RUN_DIR = backup_pointer_run("smoke")
print(SMOKE_RUN_DIR)

2026-07-22 04:27:54.949340: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-22 04:27:55.027372: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO: Runtime: Python 3.12.13 | CUDA available | GPU NVIDIA L4
INFO: Run directory: runs/20260722_042759_smoke_test_ppo_seed0
<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
<frozen importlib._bootstrap_external>:1301: Fu

## 7. Close Phase 1 with the exact IDM held-out test

This is intentionally different from the already-passing 10-episode validation reproducibility gate. The comparison command will refuse any stale incompatible baseline.

In [10]:
!python -m scripts.evaluate_baseline --config configs/ppo_mvp.yaml --policy idm --split test --episodes {PHASE1_TEST_EPISODES} --prefix idm_test
require_shell_success("Exact Phase-1 IDM test")
PHASE1_IDM_RUN_DIR = backup_pointer_run("idm")
show_json(PHASE1_IDM_RUN_DIR / "eval" / "idm_test_summary.json")

!python -m scripts.compare_runs --phase1
require_shell_success("Fingerprint-checked Phase-1 comparison")
for name in ["phase1_comparison.csv", "phase1_comparison.json", "phase1_comparison.png", "phase1_training_returns.png", "phase1_compare.log"]:
    path = REPO_DIR / "runs" / name
    if path.exists():
        copy_to_drive(path)
result_macros = REPO_DIR / "reports" / "generated_phase1_results.tex"
if result_macros.exists():
    copy_to_drive(result_macros, category="reports")

2026-07-22 04:28:32.669054: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-22 04:28:32.737732: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO: Runtime: Python 3.12.13 | CUDA available | GPU NVIDIA L4
INFO: Run directory: runs/20260722_042836_idm_baseline_seed0
INFO: Evaluating IDMPolicy for 20 episodes on the test split beginning at seed 4000.
<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cu

## 8. Verify the corrected Phase-1 video API

In [11]:
PHASE1_PPO_RUN_DIR = read_latest_run("ppo")
PHASE1_SAC_RUN_DIR = read_latest_run("sac")
PPO_VIDEO_SEED = representative_seed(PHASE1_PPO_RUN_DIR)
SAC_VIDEO_SEED = representative_seed(PHASE1_SAC_RUN_DIR)
!python -m scripts.record_video --run-dir "{PHASE1_PPO_RUN_DIR}" --model best --seed {PPO_VIDEO_SEED} --steps {VIDEO_STEPS}
require_shell_success("PPO video")
copy_to_drive(PHASE1_PPO_RUN_DIR)
!python -m scripts.record_video --run-dir "{PHASE1_SAC_RUN_DIR}" --model best --seed {SAC_VIDEO_SEED} --steps {VIDEO_STEPS}
require_shell_success("SAC video")
copy_to_drive(PHASE1_SAC_RUN_DIR)

2026-07-22 04:28:59.541887: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-22 04:28:59.617031: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO: Runtime: Python 3.12.13 | CUDA available | GPU NVIDIA L4
<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use the cuda.bindings.runtime module instead.
INFO: Loading best model on cpu for deterministic scenario seed 4000: /content/safedrive/runs/20260722_012443_

PosixPath('/content/drive/MyDrive/SafeDrive/runs/20260722_012638_sac_phase1_control_sac_seed0')

## 9. Phase-2 task-solvability checks

IDM is informative; ExpertPolicy is the feasibility gate. These use validation scenarios and do not consume the untouched test split.

In [12]:
!python -m scripts.evaluate_baseline --config configs/sac_phase2_direct.yaml --policy idm --split validation --episodes 25 --prefix phase2_idm_validation
require_shell_success("Phase-2 IDM feasibility check")
PHASE2_IDM_VALIDATION_RUN = newest_run_containing("idm_baseline")
copy_to_drive(PHASE2_IDM_VALIDATION_RUN)
show_json(PHASE2_IDM_VALIDATION_RUN / "eval" / "phase2_idm_validation_summary.json")

!python -m scripts.evaluate_baseline --config configs/sac_phase2_direct.yaml --policy expert --split validation --episodes 25 --prefix phase2_expert_validation
require_shell_success("Phase-2 ExpertPolicy feasibility check")
PHASE2_EXPERT_VALIDATION_RUN = newest_run_containing("expert_baseline")
copy_to_drive(PHASE2_EXPERT_VALIDATION_RUN)
expert_summary = show_json(PHASE2_EXPERT_VALIDATION_RUN / "eval" / "phase2_expert_validation_summary.json")
if not require_metrics(expert_summary, "Phase-2 ExpertPolicy feasibility"):
    raise RuntimeError("ExpertPolicy did not solve the target task. Inspect horizon and generated maps before training SAC.")

2026-07-22 04:29:26.368975: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2026-07-22 04:29:26.436133: I tensorflow/core/platform/cpu_feature_guard.cc:210] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
INFO: Runtime: Python 3.12.13 | CUDA available | GPU NVIDIA L4
INFO: Run directory: runs/20260722_042930_idm_baseline_seed0
INFO: Evaluating IDMPolicy for 25 episodes on the validation split beginning at seed 20000.
<frozen importlib._bootstrap_external>:1301: FutureWarning: The cuda.cudart module is deprecated and will be removed in a future release, please switch to use

RuntimeError: Phase-2 IDM feasibility check failed with exit code 1.

## 10. Seed-0 direct SAC pilot

In [ ]:
!python -m scripts.train --config configs/sac_phase2_direct.yaml --seed 0 --total-timesteps {PHASE2_TIMESTEPS}
require_shell_success("Seed-0 direct SAC training")
DIRECT_SEED0_RUN = backup_pointer_run("sac_phase2_direct_seed0")
!python -m scripts.evaluate --run-dir "{DIRECT_SEED0_RUN}" --model best --split test --episodes {PHASE2_TEST_EPISODES} --prefix best_test
require_shell_success("Seed-0 direct SAC test")
copy_to_drive(DIRECT_SEED0_RUN)
DIRECT_SEED0_SUMMARY = show_json(DIRECT_SEED0_RUN / "eval" / "best_test_summary.json")

## 11. Seed-0 curriculum SAC: curve stage

The large replay buffer is copied to Drive because the next stage resumes it. The lightweight Mac sync will skip it.

In [ ]:
!python -m scripts.train_curriculum --config configs/sac_phase2_curriculum.yaml --seed 0 --stop-after-stage curve
require_shell_success("Seed-0 curriculum curve stage")
CURRICULUM_SEED0_RUN = backup_pointer_run("sac_phase2_curriculum_seed0")
require_curriculum_state(CURRICULUM_SEED0_RUN, {"paused"})

## 11.1 Seed-0 curriculum SAC: straight-plus-curve stage

In [ ]:
CURRICULUM_SEED0_RUN = read_latest_run("sac_phase2_curriculum_seed0")
!python -m scripts.train_curriculum --config configs/sac_phase2_curriculum.yaml --run-dir "{CURRICULUM_SEED0_RUN}" --seed 0 --stop-after-stage straight_curve
require_shell_success("Seed-0 curriculum SC stage")
copy_to_drive(CURRICULUM_SEED0_RUN)
require_curriculum_state(CURRICULUM_SEED0_RUN, {"paused"})

## 11.2 Seed-0 curriculum SAC: procedural target and held-out test

In [ ]:
CURRICULUM_SEED0_RUN = read_latest_run("sac_phase2_curriculum_seed0")
!python -m scripts.train_curriculum --config configs/sac_phase2_curriculum.yaml --run-dir "{CURRICULUM_SEED0_RUN}" --seed 0
require_shell_success("Seed-0 curriculum procedural stage")
copy_to_drive(CURRICULUM_SEED0_RUN)
require_curriculum_state(CURRICULUM_SEED0_RUN, {"complete"})
!python -m scripts.evaluate --run-dir "{CURRICULUM_SEED0_RUN}" --model best --split test --episodes {PHASE2_TEST_EPISODES} --prefix best_test
require_shell_success("Seed-0 curriculum SAC test")
copy_to_drive(CURRICULUM_SEED0_RUN)
CURRICULUM_SEED0_SUMMARY = show_json(CURRICULUM_SEED0_RUN / "eval" / "best_test_summary.json")

## 12. Compare seed-0 pilots and decide whether confirmation is justified

In [ ]:
!python -m scripts.compare_runs --phase2 --seeds 0
require_shell_success("Seed-0 Phase-2 comparison")
PILOTS_PROMISING = any(float(summary.get("success_rate", 0.0)) >= 0.50 or float(summary.get("mean_route_completion", 0.0)) >= 0.80 for summary in [DIRECT_SEED0_SUMMARY, CURRICULUM_SEED0_SUMMARY])
print(f"Confirmation justified: {PILOTS_PROMISING}")
if not PILOTS_PROMISING:
    print("Stop here. Do not spend credits on seeds 1 and 2; inspect the seed-0 diagnostics first.")
for name in ["phase2_seed_results.csv", "phase2_comparison.csv", "phase2_comparison.json", "phase2_comparison.png", "phase2_training_returns.png", "phase2_compare.log"]:
    path = REPO_DIR / "runs" / name
    if path.exists():
        copy_to_drive(path)

## 13. Conditional confirmation: seed 1

Run this section only when the previous cell prints `Confirmation justified: True`. The curriculum command pauses and backs up at both prerequisite stages inside this cell.

In [ ]:
assert PILOTS_PROMISING, "Seed-0 pilots did not justify confirmation runs."
!python -m scripts.train --config configs/sac_phase2_direct.yaml --seed 1 --total-timesteps {PHASE2_TIMESTEPS}
require_shell_success("Seed-1 direct SAC training")
DIRECT_SEED1_RUN = backup_pointer_run("sac_phase2_direct_seed1")
!python -m scripts.evaluate --run-dir "{DIRECT_SEED1_RUN}" --model best --split test --episodes {PHASE2_TEST_EPISODES} --prefix best_test
require_shell_success("Seed-1 direct SAC test")
copy_to_drive(DIRECT_SEED1_RUN)

!python -m scripts.train_curriculum --config configs/sac_phase2_curriculum.yaml --seed 1 --stop-after-stage curve
require_shell_success("Seed-1 curriculum curve stage")
CURRICULUM_SEED1_RUN = backup_pointer_run("sac_phase2_curriculum_seed1")
require_curriculum_state(CURRICULUM_SEED1_RUN, {"paused"})
!python -m scripts.train_curriculum --config configs/sac_phase2_curriculum.yaml --run-dir "{CURRICULUM_SEED1_RUN}" --seed 1 --stop-after-stage straight_curve
require_shell_success("Seed-1 curriculum SC stage")
copy_to_drive(CURRICULUM_SEED1_RUN)
require_curriculum_state(CURRICULUM_SEED1_RUN, {"paused"})
!python -m scripts.train_curriculum --config configs/sac_phase2_curriculum.yaml --run-dir "{CURRICULUM_SEED1_RUN}" --seed 1
require_shell_success("Seed-1 curriculum procedural stage")
copy_to_drive(CURRICULUM_SEED1_RUN)
require_curriculum_state(CURRICULUM_SEED1_RUN, {"complete"})
!python -m scripts.evaluate --run-dir "{CURRICULUM_SEED1_RUN}" --model best --split test --episodes {PHASE2_TEST_EPISODES} --prefix best_test
require_shell_success("Seed-1 curriculum SAC test")
copy_to_drive(CURRICULUM_SEED1_RUN)

## 13.1 Conditional confirmation: seed 2

In [ ]:
assert PILOTS_PROMISING, "Seed-0 pilots did not justify confirmation runs."
!python -m scripts.train --config configs/sac_phase2_direct.yaml --seed 2 --total-timesteps {PHASE2_TIMESTEPS}
require_shell_success("Seed-2 direct SAC training")
DIRECT_SEED2_RUN = backup_pointer_run("sac_phase2_direct_seed2")
!python -m scripts.evaluate --run-dir "{DIRECT_SEED2_RUN}" --model best --split test --episodes {PHASE2_TEST_EPISODES} --prefix best_test
require_shell_success("Seed-2 direct SAC test")
copy_to_drive(DIRECT_SEED2_RUN)

!python -m scripts.train_curriculum --config configs/sac_phase2_curriculum.yaml --seed 2 --stop-after-stage curve
require_shell_success("Seed-2 curriculum curve stage")
CURRICULUM_SEED2_RUN = backup_pointer_run("sac_phase2_curriculum_seed2")
require_curriculum_state(CURRICULUM_SEED2_RUN, {"paused"})
!python -m scripts.train_curriculum --config configs/sac_phase2_curriculum.yaml --run-dir "{CURRICULUM_SEED2_RUN}" --seed 2 --stop-after-stage straight_curve
require_shell_success("Seed-2 curriculum SC stage")
copy_to_drive(CURRICULUM_SEED2_RUN)
require_curriculum_state(CURRICULUM_SEED2_RUN, {"paused"})
!python -m scripts.train_curriculum --config configs/sac_phase2_curriculum.yaml --run-dir "{CURRICULUM_SEED2_RUN}" --seed 2
require_shell_success("Seed-2 curriculum procedural stage")
copy_to_drive(CURRICULUM_SEED2_RUN)
require_curriculum_state(CURRICULUM_SEED2_RUN, {"complete"})
!python -m scripts.evaluate --run-dir "{CURRICULUM_SEED2_RUN}" --model best --split test --episodes {PHASE2_TEST_EPISODES} --prefix best_test
require_shell_success("Seed-2 curriculum SAC test")
copy_to_drive(CURRICULUM_SEED2_RUN)

## 14. Exact Phase-2 IDM and ExpertPolicy test baselines

In [ ]:
!python -m scripts.evaluate_baseline --config configs/sac_phase2_direct.yaml --policy idm --split test --episodes {PHASE2_TEST_EPISODES} --prefix phase2_idm_test
require_shell_success("Phase-2 IDM test")
PHASE2_IDM_RUN = backup_pointer_run("phase2_idm")
!python -m scripts.evaluate_baseline --config configs/sac_phase2_direct.yaml --policy expert --split test --episodes {PHASE2_TEST_EPISODES} --prefix phase2_expert_test
require_shell_success("Phase-2 ExpertPolicy test")
PHASE2_EXPERT_RUN = backup_pointer_run("phase2_expert")

## 15. Zero-shot light-traffic stress test

In [ ]:
DIRECT_SEED0_RUN = read_latest_run("sac_phase2_direct_seed0")
CURRICULUM_SEED0_RUN = read_latest_run("sac_phase2_curriculum_seed0")
!python -m scripts.evaluate --run-dir "{DIRECT_SEED0_RUN}" --model best --split test --episodes {PHASE2_TEST_EPISODES} --prefix best_light_traffic --overwrite test.traffic_density=0.05 test.random_traffic=true
require_shell_success("Seed-0 direct light-traffic test")
copy_to_drive(DIRECT_SEED0_RUN)
!python -m scripts.evaluate --run-dir "{CURRICULUM_SEED0_RUN}" --model best --split test --episodes {PHASE2_TEST_EPISODES} --prefix best_light_traffic --overwrite test.traffic_density=0.05 test.random_traffic=true
require_shell_success("Seed-0 curriculum light-traffic test")
copy_to_drive(CURRICULUM_SEED0_RUN)
if PILOTS_PROMISING:
    print("Run the next optional cell for seeds 1 and 2 before the final comparison.")

### Optional confirmation light-traffic evaluations

In [ ]:
assert PILOTS_PROMISING, "Confirmation runs were not justified."
DIRECT_SEED1_RUN = read_latest_run("sac_phase2_direct_seed1")
CURRICULUM_SEED1_RUN = read_latest_run("sac_phase2_curriculum_seed1")
DIRECT_SEED2_RUN = read_latest_run("sac_phase2_direct_seed2")
CURRICULUM_SEED2_RUN = read_latest_run("sac_phase2_curriculum_seed2")
!python -m scripts.evaluate --run-dir "{DIRECT_SEED1_RUN}" --model best --split test --episodes {PHASE2_TEST_EPISODES} --prefix best_light_traffic --overwrite test.traffic_density=0.05 test.random_traffic=true
require_shell_success("Seed-1 direct light traffic")
copy_to_drive(DIRECT_SEED1_RUN)
!python -m scripts.evaluate --run-dir "{CURRICULUM_SEED1_RUN}" --model best --split test --episodes {PHASE2_TEST_EPISODES} --prefix best_light_traffic --overwrite test.traffic_density=0.05 test.random_traffic=true
require_shell_success("Seed-1 curriculum light traffic")
copy_to_drive(CURRICULUM_SEED1_RUN)
!python -m scripts.evaluate --run-dir "{DIRECT_SEED2_RUN}" --model best --split test --episodes {PHASE2_TEST_EPISODES} --prefix best_light_traffic --overwrite test.traffic_density=0.05 test.random_traffic=true
require_shell_success("Seed-2 direct light traffic")
copy_to_drive(DIRECT_SEED2_RUN)
!python -m scripts.evaluate --run-dir "{CURRICULUM_SEED2_RUN}" --model best --split test --episodes {PHASE2_TEST_EPISODES} --prefix best_light_traffic --overwrite test.traffic_density=0.05 test.random_traffic=true
require_shell_success("Seed-2 curriculum light traffic")
copy_to_drive(CURRICULUM_SEED2_RUN)

## 16. Final strict comparison and representative videos

In [ ]:
COMPARISON_SEEDS = [0, 1, 2] if PILOTS_PROMISING else [0]
subprocess.run([sys.executable, "-m", "scripts.compare_runs", "--phase2", "--seeds", *[str(seed) for seed in COMPARISON_SEEDS]], cwd=REPO_DIR, check=True)
for path in (REPO_DIR / "runs").glob("phase2_*"):
    if path.is_file():
        copy_to_drive(path)

DIRECT_VIDEO_SEED = representative_seed(DIRECT_SEED0_RUN)
CURRICULUM_VIDEO_SEED = representative_seed(CURRICULUM_SEED0_RUN)
!python -m scripts.record_video --run-dir "{DIRECT_SEED0_RUN}" --model best --seed {DIRECT_VIDEO_SEED} --steps {VIDEO_STEPS}
require_shell_success("Direct SAC video")
copy_to_drive(DIRECT_SEED0_RUN)
!python -m scripts.record_video --run-dir "{CURRICULUM_SEED0_RUN}" --model best --seed {CURRICULUM_VIDEO_SEED} --steps {VIDEO_STEPS}
require_shell_success("Curriculum SAC video")
copy_to_drive(CURRICULUM_SEED0_RUN)

## 17. Build reports and perform final artifact sync

In [ ]:
if shutil.which("latexmk"):
    subprocess.run(["latexmk", "-cd", "-pdf", "-interaction=nonstopmode", "-halt-on-error", "reports/main.tex"], cwd=REPO_DIR, check=True)
    subprocess.run(["latexmk", "-cd", "-pdf", "-interaction=nonstopmode", "-halt-on-error", "reports/surrogate_notes.tex"], cwd=REPO_DIR, check=True)
else:
    print("latexmk is unavailable in this runtime; compile the reports locally.")
(DRIVE_PROJECT / "runs").mkdir(parents=True, exist_ok=True)
(DRIVE_PROJECT / "reports").mkdir(parents=True, exist_ok=True)
if shutil.which("rsync"):
    subprocess.run(["rsync", "-a", f"{REPO_DIR / 'runs'}/", f"{DRIVE_PROJECT / 'runs'}/"], check=True)
    subprocess.run(["rsync", "-a", f"{REPO_DIR / 'reports'}/", f"{DRIVE_PROJECT / 'reports'}/"], check=True)
else:
    shutil.copytree(REPO_DIR / "runs", DRIVE_PROJECT / "runs", dirs_exist_ok=True)
    shutil.copytree(REPO_DIR / "reports", DRIVE_PROJECT / "reports", dirs_exist_ok=True)
print(f"Final artifacts synchronized to {DRIVE_PROJECT}")

## Finished

Disconnect the Colab runtime when complete. On the Mac, run `python -m scripts.sync_drive_runs` to bring back only analysis artifacts, logs, plots, summaries, videos, and reports.